# Layers MART — Feature Engineering, Estrella BI y Modelo ML

**Objetivo:** A partir de la capa STG, construir:
- **MART_ML**: tabla analítica con features para el modelo de machine learning.
- **MART_BI**: modelo estrella (fact + dimensiones) listo para Looker/PowerBI.
- **Modelo ML**: predicción de duración de recorrido con scikit-learn.

## Modelo estrella MART_BI
```
              dim_fecha
                  │
dim_usuario ──► fact_recorridos ◄── dim_estacion
                  │
              dim_bicicleta
```

> **Nota de rendimiento:** el notebook libera memoria (`del` + `gc.collect()`) entre pasos pesados para evitar crasheos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_theme(style='whitegrid')

# Path().resolve() en Jupyter apunta al directorio del notebook (notebooks/).
# .parent sube al root del proyecto. Funciona en cualquier maquina sin cambiar nada.
BASE_DIR = Path().resolve().parent
STG_DIR  = BASE_DIR / "data" / "stg"
MART_ML  = BASE_DIR / "data" / "mart_ml"
MART_BI  = BASE_DIR / "data" / "mart_bi"
MART_ML.mkdir(parents=True, exist_ok=True)
MART_BI.mkdir(parents=True, exist_ok=True)

print("Entorno configurado.")
print(f"  BASE_DIR: {BASE_DIR}")

## 1. Carga STG — solo columnas necesarias

In [ ]:
COLS_REC = [
    'id_recorrido', 'id_usuario', 'duracion_recorrido',
    'fecha_origen_recorrido', 'fecha_destino_recorrido',
    'id_estacion_origen',  'nombre_estacion_origen',
    'direccion_estacion_origen', 'lat_estacion_origen', 'long_estacion_origen',
    'id_estacion_destino', 'nombre_estacion_destino',
    'direccion_estacion_destino', 'lat_estacion_destino', 'long_estacion_destino',
    'modelo_bicicleta', 'genero', 'anio'
]

rec = pd.read_parquet(STG_DIR / "recorridos.parquet",
                      columns=[c for c in COLS_REC])

for col in ['id_estacion_origen', 'id_estacion_destino', 'duracion_recorrido', 'anio']:
    if col in rec.columns:
        rec[col] = pd.to_numeric(rec[col], errors='coerce').astype('float32')
for col in ['lat_estacion_origen', 'long_estacion_origen',
            'lat_estacion_destino', 'long_estacion_destino']:
    if col in rec.columns:
        rec[col] = pd.to_numeric(rec[col], errors='coerce').astype('float32')

print(f"Recorridos STG: {rec.shape}")
print(f"RAM estimada  : {rec.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

## 2. Feature Engineering

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return (R * 2 * np.arcsin(np.sqrt(a))).astype('float32')

rec['hora']         = rec['fecha_origen_recorrido'].dt.hour.astype('int8')
rec['dia_semana']   = rec['fecha_origen_recorrido'].dt.dayofweek.astype('int8')
rec['mes']          = rec['fecha_origen_recorrido'].dt.month.astype('int8')
rec['trimestre']    = rec['fecha_origen_recorrido'].dt.quarter.astype('int8')
rec['es_finde']     = (rec['dia_semana'] >= 5).astype('int8')
rec['es_hora_pico'] = rec['hora'].isin([7, 8, 9, 17, 18, 19]).astype('int8')
rec['id_fecha']     = rec['fecha_origen_recorrido'].dt.strftime('%Y%m%d').astype('Int64')

rec['distancia_km'] = haversine_km(
    rec['lat_estacion_origen'].values,  rec['long_estacion_origen'].values,
    rec['lat_estacion_destino'].values, rec['long_estacion_destino'].values
)
rec['modelo_enc']  = (rec['modelo_bicicleta'] == 'FIT').astype('int8')
rec['genero_enc']  = rec['genero'].map({'MALE': 0, 'FEMALE': 1, 'OTHER': 2}).astype('float32')
rec['duracion_min']= (rec['duracion_recorrido'] / 60).astype('float32')

gc.collect()
print(f"Features calculadas. RAM: {rec.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

## 3. MART_ML

In [ ]:
FEATURES = ['hora', 'dia_semana', 'mes', 'trimestre', 'es_finde',
            'es_hora_pico', 'anio', 'modelo_enc', 'genero_enc', 'distancia_km']
TARGET   = 'duracion_recorrido'

mart_ml = rec[FEATURES + [TARGET, 'id_recorrido']].dropna(subset=[TARGET]).reset_index(drop=True)
mart_ml.to_parquet(MART_ML / "features_recorridos.parquet", index=False, compression='snappy')
print(f"MART_ML guardado: {mart_ml.shape}")

## 4. MART_BI — Modelo estrella

In [ ]:
# --- dim_fecha ---
fechas = rec['fecha_origen_recorrido'].dropna().dt.normalize().unique()
dim_fecha = pd.DataFrame({'fecha': pd.DatetimeIndex(fechas)})
dim_fecha['id_fecha']   = dim_fecha['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_fecha['anio']       = dim_fecha['fecha'].dt.year.astype('int16')
dim_fecha['mes']        = dim_fecha['fecha'].dt.month.astype('int8')
dim_fecha['dia']        = dim_fecha['fecha'].dt.day.astype('int8')
dim_fecha['trimestre']  = dim_fecha['fecha'].dt.quarter.astype('int8')
dim_fecha['dia_semana'] = dim_fecha['fecha'].dt.dayofweek.astype('int8')
dim_fecha['nombre_dia'] = dim_fecha['fecha'].dt.day_name()
dim_fecha['nombre_mes'] = dim_fecha['fecha'].dt.month_name()
dim_fecha['es_finde']   = (dim_fecha['dia_semana'] >= 5).astype('int8')
dim_fecha = dim_fecha.sort_values('fecha').reset_index(drop=True)
dim_fecha.to_parquet(MART_BI / "dim_fecha.parquet", index=False)
print(f"dim_fecha: {dim_fecha.shape}")
del dim_fecha; gc.collect()

In [ ]:
# --- dim_estacion ---
est_o = rec[['id_estacion_origen','nombre_estacion_origen',
             'direccion_estacion_origen','lat_estacion_origen','long_estacion_origen']].copy()
est_o.columns = ['id_estacion','nombre','direccion','lat','lon']

est_d = rec[['id_estacion_destino','nombre_estacion_destino',
             'direccion_estacion_destino','lat_estacion_destino','long_estacion_destino']].copy()
est_d.columns = ['id_estacion','nombre','direccion','lat','lon']

dim_estacion = (
    pd.concat([est_o, est_d], ignore_index=True)
    .dropna(subset=['id_estacion'])
    .drop_duplicates(subset=['id_estacion'])
    .sort_values('id_estacion')
    .reset_index(drop=True)
)
dim_estacion.to_parquet(MART_BI / "dim_estacion.parquet", index=False)
print(f"dim_estacion: {dim_estacion.shape}")
del est_o, est_d, dim_estacion; gc.collect()

In [ ]:
# --- dim_usuario ---
usr = pd.read_parquet(STG_DIR / "usuarios.parquet")
if 'edad_usuario' in usr.columns:
    usr['edad_usuario'] = pd.to_numeric(usr['edad_usuario'], errors='coerce')
    usr['rango_edad'] = pd.cut(
        usr['edad_usuario'],
        bins=[0, 17, 25, 35, 50, 120],
        labels=['<18', '18-25', '26-35', '36-50', '50+']
    ).astype(str)
usr.to_parquet(MART_BI / "dim_usuario.parquet", index=False)
print(f"dim_usuario: {usr.shape}")
del usr; gc.collect()

In [ ]:
# --- dim_bicicleta ---
dim_bicicleta = pd.DataFrame({
    'modelo_bicicleta': ['FIT', 'ICONIC'],
    'tipo': ['Electrica asistida', 'Clasica mecanica'],
    'descripcion': [
        'Bicicleta con asistencia electrica, ideal para distancias largas',
        'Bicicleta clasica sin asistencia electrica'
    ]
})
dim_bicicleta.to_parquet(MART_BI / "dim_bicicleta.parquet", index=False)
print(f"dim_bicicleta: {dim_bicicleta.shape}")
del dim_bicicleta; gc.collect()

In [ ]:
# --- fact_recorridos ---
FACT_COLS = ['id_recorrido','id_usuario','id_estacion_origen','id_estacion_destino',
             'id_fecha','duracion_recorrido','duracion_min','modelo_bicicleta','genero',
             'hora','dia_semana','mes','anio','es_finde','distancia_km']
fact = rec[[c for c in FACT_COLS if c in rec.columns]].copy()
fact.to_parquet(MART_BI / "fact_recorridos.parquet", index=False, compression='snappy')
print(f"fact_recorridos: {fact.shape}")
del fact; gc.collect()

print("\n=== RESUMEN MART_BI ===")
for f in sorted(MART_BI.glob("*.parquet")):
    df_tmp = pd.read_parquet(f)
    mb = f.stat().st_size / 1024**2
    print(f"  {f.name}: {df_tmp.shape[0]:,} filas x {df_tmp.shape[1]} cols  ({mb:.1f} MB)")
    del df_tmp
gc.collect()

## 5. Modelo de Machine Learning

**Problema:** Predicción de duración de recorrido (regresión).

Usamos una **muestra de 100 000 filas** para que el entrenamiento sea viable en cualquier PC.

In [ ]:
del rec; gc.collect()

mart_df = pd.read_parquet(MART_ML / "features_recorridos.parquet")

FEATURES = ['hora', 'dia_semana', 'mes', 'trimestre', 'es_finde',
            'es_hora_pico', 'anio', 'modelo_enc', 'genero_enc', 'distancia_km']
TARGET = 'duracion_recorrido'

N_SAMPLE = min(100_000, len(mart_df))
sample = mart_df.dropna(subset=FEATURES + [TARGET]).sample(n=N_SAMPLE, random_state=42)
del mart_df; gc.collect()

X = sample[FEATURES].astype('float32')
y = sample[TARGET].astype('float32')
del sample; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train):,} filas | Test: {len(X_test):,} filas")

In [ ]:
# --- Modelo 1: Regresión Lineal (baseline) ---
pipe_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LinearRegression())
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)
print(f"LinearRegression — MAE: {mae_lr:.0f}s ({mae_lr/60:.1f} min) | RMSE: {rmse_lr:.0f}s | R²: {r2_lr:.4f}")

In [ ]:
# --- Modelo 2: Random Forest ---
pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model',   RandomForestRegressor(
                    n_estimators=50,
                    max_depth=8,
                    min_samples_leaf=20,
                    random_state=42,
                    n_jobs=-1
                ))
])
print("Entrenando Random Forest...")
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)
print(f"RandomForest     — MAE: {mae_rf:.0f}s ({mae_rf/60:.1f} min) | RMSE: {rmse_rf:.0f}s | R²: {r2_rf:.4f}")

In [ ]:
resultados = pd.DataFrame({
    'Modelo':     ['Regresión Lineal', 'Random Forest'],
    'MAE (seg)':  [round(mae_lr), round(mae_rf)],
    'MAE (min)':  [round(mae_lr/60, 1), round(mae_rf/60, 1)],
    'RMSE (seg)': [round(rmse_lr), round(rmse_rf)],
    'R²':         [round(r2_lr, 4), round(r2_rf, 4)]
})
print(resultados.to_string(index=False))
mejor = resultados.loc[resultados['R²'].idxmax(), 'Modelo']
print(f"\nMejor modelo: {mejor}")

In [ ]:
# --- Importancia de features ---
importancias = pd.Series(
    pipe_rf.named_steps['model'].feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importancias.plot(kind='barh', ax=ax, color=sns.color_palette('Blues_r', len(FEATURES)))
ax.set_title('Importancia de features — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout()
plt.savefig(BASE_DIR / 'notebooks' / 'viz5_feature_importance.png', dpi=100)
plt.show()

In [ ]:
# --- Real vs Predicho ---
idx = np.random.choice(len(y_test), size=min(3000, len(y_test)), replace=False)
yt  = np.array(y_test)[idx]
yp  = y_pred_rf[idx]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(yt/60, yp/60, alpha=0.2, s=8, color='steelblue')
lim = max(yt.max(), yp.max()) / 60
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Predicción perfecta')
ax.set_xlabel('Duración real (min)')
ax.set_ylabel('Duración predicha (min)')
ax.set_title(f'Real vs Predicho — Random Forest  (R²={r2_rf:.3f})', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'notebooks' / 'viz6_real_vs_predicho.png', dpi=100)
plt.show()

In [ ]:
pd.DataFrame({
    'duracion_real_seg': np.array(y_test),
    'duracion_pred_lr_seg': y_pred_lr,
    'duracion_pred_rf_seg': y_pred_rf,
}).to_parquet(MART_ML / "predicciones_test.parquet", index=False)

print("=== PIPELINE COMPLETO ===")
print(f"Mejor modelo : {mejor}")
print(f"MAE          : {mae_rf:.0f} seg  ({mae_rf/60:.1f} min)")
print(f"RMSE         : {rmse_rf:.0f} seg")
print(f"R²           : {r2_rf:.4f}")
print("\nTodos los Parquet guardados en data/mart_ml/ y data/mart_bi/")